# 🍅 Tomato-Oversight — 학습 루프 (Colab + Drive + GitHub)

**로컬(VS Code)에서 코드 수정 → git push → 여기서 pull → GPU 학습 → 결과는 Drive → 로그 분석** 이 한 바퀴를 도는 노트북입니다.

- **위에서부터 순서대로 실행.** 매 세션 1~4는 한 번씩, 5~7은 학습할 때마다.
- 바꾸는 곳은 **아래 `설정` 셀 하나뿐** — `FOLDER`(cheater_v1 / honest_v11), `RUN`, `TOTAL_STEPS`만 고치면 됩니다.
- 결과(.pt·csv·summary)는 전부 **Google Drive**에 저장돼 런타임이 꺼져도 보존됩니다.
- GPU는 **T4** 권장(이 작업엔 A100·L4 이득 없음, 크레딧만 소모).


## 0. 설정 — 여기만 수정

`FOLDER`를 학습할 폴더 이름으로 두면 나머지 경로가 전부 자동으로 맞춰집니다.

In [ ]:
# ===== 여기만 바꾸세요 =====
FOLDER      = "cheater_v1"     # "cheater_v1" 또는 "honest_v11"
RUN         = "run1"           # 실험 구분용 이름 (run1, run2, sweep_lr ...)
TOTAL_STEPS = 500_000          # 이 과제는 200k~500k면 수렴 (1M은 낭비)
# ==========================

REPO_URL = "https://github.com/1ee1ee1ee/tomato-oversight.git"
REPO_DIR = "/content/tomato-oversight"
WORKDIR  = f"{REPO_DIR}/{FOLDER}"
OUTPUT_DIR = f"/content/drive/MyDrive/result_{FOLDER}/{RUN}"   # 결과 저장 위치(Drive)
BEST_MODEL = f"{OUTPUT_DIR}/{FOLDER}_best.pt"

print("학습 폴더 :", WORKDIR)
print("결과 저장 :", OUTPUT_DIR)
print("best 모델 :", BEST_MODEL)

## 1. Google Drive 마운트

결과를 Drive에 저장하기 위해 연결합니다. (팝업 인증 1회)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 코드 최신화 (GitHub → Colab)

처음이면 clone, 이미 있으면 `git pull`로 로컬에서 push한 최신 코드를 당겨옵니다.

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --no-rebase

print("\n현재 커밋:")
!git -C {REPO_DIR} log -1 --oneline
print("학습 폴더 존재:", os.path.isdir(WORKDIR))

## 3. 패키지 설치 + 테스트

의존성 설치 후 단위 테스트를 돌려 환경 규칙이 안 깨졌는지 먼저 확인합니다.

In [ ]:
!cd "{WORKDIR}" && pip install -q -r requirements.txt
!cd "{WORKDIR}" && python -m unittest discover -s tests -v

## 4. GPU 확인

`런타임 > 런타임 유형 변경`에서 **T4 GPU** 선택. 아래가 `True`여야 합니다.

In [ ]:
import torch
print("GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")

## 5. 학습 실행 (결과는 Drive로)

`--output-dir`을 Drive로 지정해 학습 중 best 모델·CSV가 실시간 저장됩니다.
런타임이 끊겨도 중간 산출물이 남습니다.

In [ ]:
cmd = (f'cd "{WORKDIR}" && python train.py '
       f'--total-steps {TOTAL_STEPS} --device auto '
       f'--output-dir "{OUTPUT_DIR}"')
print(cmd, "\n")
!{cmd}

print("\n=== 저장된 파일 ===")
!ls -lh "{OUTPUT_DIR}"

## 6. 평가 (Drive의 best 모델 로드)

재학습 없이 저장된 best 모델을 불러와 배포 정책(ε-greedy 0.10)으로 평가합니다.

In [ ]:
cmd = (f'cd "{WORKDIR}" && python evaluate.py '
       f'--model "{BEST_MODEL}" --episodes-per-o 10 --device auto')
print(cmd, "\n")
!{cmd}

## 7. 로그 요약 (분석용 — 이 출력을 그대로 붙여넣기)

`summary.json`(최종 지표)과 `periodic_evaluation.csv` 끝부분(수렴 추이)을 출력합니다.
**이 셀 출력을 복사해서 Claude에게 주면** 붕괴·수렴·지표를 분석해 코드/파라미터 수정안을 받습니다.

In [ ]:
import json, pathlib

print("========== summary.json ==========")
p = pathlib.Path(OUTPUT_DIR) / "summary.json"
if p.exists():
    print(json.dumps(json.loads(p.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("(아직 없음 — 학습이 끝나면 생성됩니다)")

print("\n===== periodic_evaluation.csv (마지막 12줄) =====")
!tail -n 12 "{OUTPUT_DIR}/periodic_evaluation.csv" 

## 다음 학습부터의 루프

1. **로컬(VS Code)** 에서 코드/파라미터 수정 → `git push`
2. **여기서** 셀 2(`git pull`) → 셀 5(학습) 실행 — `FOLDER`/`RUN`만 맞추면 됨
3. **셀 7 출력을 Claude에 붙여넣기** → 분석·수정안 받기 → 1로 반복

> ⚠️ 모델 `.pt`는 Drive에만(용량 큼). GitHub엔 코드·문서·작은 CSV만.
> ⚠️ Colab에서 코드를 직접 고쳤다면 push 전 로컬과 충돌 확인.